# Demonstration of S1 WV and SWOT KaRin Matchups

This notebook illustrates the sequence of steps to identify and generate matchups between Sentinel-1 WindWave (WV) products and SWOT KaRin swath data.

In [ ]:
import os
import xarray as xr
import numpy as np
from s1swotcolocs.generate_wv_karin_matchups_stac_items import (\n    parse_wv_metadata, \n    parse_swot_metadata, \n    is_match, \n    generate_stac_item\n)

## 1. Load Asset Files

We use the minimal fixtures provided in the library assets directory.

In [ ]:
import s1swotcolocs
assets_dir = os.path.join(os.path.dirname(s1swotcolocs.__file__), "assets")

S1_FILE = os.path.join(assets_dir, "test_s1_wv_minimal.nc")
SWOT_FILE = os.path.join(assets_dir, "test_swot_l2_minimal.nc")

print(f"S1 file: {S1_FILE}")
print(f"SWOT file: {SWOT_FILE}")

## 2. Extract Metadata

The first step is to extract temporal and spatial information (footprints) from the NetCDF files.

In [ ]:
s1_meta = parse_wv_metadata(S1_FILE)
swot_meta = parse_swot_metadata(SWOT_FILE)

print("S1 Metadata:", s1_meta)
print("\nSWOT Metadata:", swot_meta)

## 3. Perform Matchup Check

A matchup is valid if the S1 timestamp falls within the SWOT swath window (with a small threshold) and their spatial footprints intersect.

In [ ]:
match_status = is_match(s1_meta, swot_meta, time_threshold_min=10)
print(f"Is match: {match_status}")

## 4. Generate STAC Item

If a match is found, we can generate a STAC (SpatioTemporal Asset Catalog) item to standardize the metadata for downstream processing.

In [ ]:
if match_status:
    stac_item = generate_stac_item(s1_meta, swot_meta)
    print("STAC Item ID:", stac_item.id)
    print("Properties:", stac_item.properties)
else:
    print("No match found; skipping STAC generation.")